In [3]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from tqdm import tqdm
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, Subset
from functools import partial

from src.data.hand_landmarks import HandLandmarksDataset
from src.models.SLT_model import SignLanguageTranslator, SignLanguageTranslatorV1
from config import ROOT

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FEATURE_DIR = os.path.join(ROOT, "datasets", "processed", "mediapipe")
BATCH_SIZE = 8
def collate_fn(batch, tokenizer):

    features, texts = [], []

    for feature, text in batch:
        features.append(feature)
        texts.append(text)

    real_lengths = [f.shape[0] for f in features]

    features = pad_sequence(features, batch_first=True)

    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    video_mask = (
        torch.arange(features.shape[1]).unsqueeze(0)
        < torch.tensor(real_lengths).unsqueeze(1)
    ).long()

    labels = texts.clone()
    labels[labels == tokenizer.pad_token_id] = -100

    return features, labels, video_mask


In [2]:
tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small",
    use_fast=False
)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [4]:
dataset = HandLandmarksDataset(FEATURE_DIR, tokenizer)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=partial(collate_fn, tokenizer=tokenizer),
    num_workers=2,
    pin_memory=True
)

In [ ]:
feature, text_ids = dataset[0]

for i in range(len(feature)):
    print(feature[i])

In [40]:
x1, _ = dataset[0]
x2, _ = dataset[1]
x3, _ = dataset[2]
x4, _ = dataset[3]

print(x1.std(), x2.std(), x3.std(), x4.std())
print(x1.mean(), x2.mean(), x3.mean(), x4.mean())

tensor(0.0442) tensor(0.0399) tensor(0.0325) tensor(0.0350)
tensor(-0.0232) tensor(-0.0135) tensor(-0.0117) tensor(-0.0098)


In [41]:
import torch.nn.functional as F

x = [dataset[i][0].mean(0) for i in range(20)]
x = torch.stack(x)

print("pairwise cos:")
for i in range(5):
    for j in range(i+1, 5):
        print(F.cosine_similarity(x[i].unsqueeze(0), x[j].unsqueeze(0)))

pairwise cos:
tensor([0.9443])
tensor([0.9169])
tensor([0.7762])
tensor([0.9740])
tensor([0.9653])
tensor([0.7185])
tensor([0.9457])
tensor([0.7663])
tensor([0.9550])
tensor([0.7711])
